In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

# Guardem el target abans de res
y = np.log1p(train['SalePrice'])

# Treiem SalePrice i Id per treballar còmode
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

# Combinem train i test per fer el preprocessing igual als dos
df = pd.concat([train, test], axis=0).reset_index(drop=True)

print(f"Dataset combinat: {df.shape}")

Dataset combinat: (2919, 79)


In [2]:
# Nuls que signifiquen "absent" → omplir amb "None" o 0

# Categòriques — "None" = no té
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
             'BsmtFinType2', 'MasVnrType']

for col in cols_none:
    df[col] = df[col].fillna('None')

# Numèriques — 0 = no té
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    df[col] = df[col].fillna(0)

print("Nuls semàntics tractats!")
print(df[cols_none + cols_zero].isnull().sum().sum(), "nuls restants en aquestes columnes")

Nuls semàntics tractats!
0 nuls restants en aquestes columnes


In [3]:
# LotFrontage — mediana per Neighborhood (recordes el brainstorm?)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage']\
                      .transform(lambda x: x.fillna(x.median()))

df['MasVnrArea'] = df.groupby('Neighborhood')['MasVnrArea']\
                     .transform(lambda x: x.fillna(x.median()))

df['Electrical'] = df.groupby('Neighborhood')['Electrical']\
                     .transform(lambda x: x.fillna(x.mode()[0]))

print("Nuls reals tractats!")
print("Nuls restants al dataset:", df.isnull().sum().sum())

Nuls reals tractats!
Nuls restants al dataset: 12


In [4]:
df.isnull().sum()[df.isnull().sum() > 0]

MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64

In [5]:
cols_moda = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 
             'KitchenQual', 'Functional', 'SaleType']

for col in cols_moda:
    df[col] = df.groupby('Neighborhood')[col]\
                .transform(lambda x: x.fillna(x.mode()[0]))

print("Nuls restants:", df.isnull().sum().sum())

Nuls restants: 0
